# 25 Perbandingan Initializer Terintegrasi


In [1]:
from pathlib import Path
import os, sys, types
from IPython.display import display

PROJECT_ROOT = '/content/drive/MyDrive/REVISED'
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', force_remount=False)
_ROOT_25 = Path(os.environ.get('INITIALIZER25_ROOT', PROJECT_ROOT)).expanduser().resolve()
if not (_ROOT_25 / 'LIB/care_km.py').is_file():
    raise FileNotFoundError('LIB/care_km.py belum tersedia pada ' + str(_ROOT_25))
sys.dont_write_bytecode = True
print('Proyek:', _ROOT_25)


Mounted at /content/drive
Proyek: /content/drive/MyDrive/REVISED


## Kelompok eksperimen dan pengulangan

| Kelompok | Data yang dipakai | Protokol |
|---|---|---|
| LMS full | 6.658 peserta, 13 fitur | k=16; 15 run per metode stokastik, 1 per deterministik |
| LMS resampling | 30 indeks 80% yang tersimpan | satu run per metode per resample; k=16 |
| LMS train | 4.660 train / 1.998 test | fit pada train; simpan assignment test; k=16 |
| Pendikom train | 322 train / 139 test, 18 fitur | fit pada train; simpan assignment test; k=16 |
| Benchmark full | sepuluh dataset pada registry | k sesuai kelas referensi; 10 run stokastik, 1 deterministik |
| Benchmark resampling | 30 indeks 80% per dataset | satu run per metode per resample; k tetap per dataset |

Untuk kelompok train, metode stokastik memakai 15 seed yang ditetapkan.
BK-Modes deterministik pada urutan data dan kebijakan tie yang dicatat.
Seluruh optimizer akhir memakai batas 100 iterasi. Tidak ada pemilihan
best run. Label benchmark dipakai untuk menentukan k dan metrik eksternal,
bukan sebagai fitur atau masukan inisialisasi.

Jalankan ulang notebook yang sama jika sesi terputus. Checkpoint per run
memeriksa kode, isi matriks/indeks, parameter, lingkungan, dan artefak.
Hasil lama digunakan hanya jika catatan isi dan protokolnya sesuai. Bila
ringkasan lama tersedia tetapi mode/assignment yang diperlukan belum ada,
fitting tersebut dilakukan sekali untuk membentuk bank partisi.


In [2]:
KONFIGURASI_25 = {
    'scopes': ['LMS_FULL', 'LMS_RESAMPLING', 'LMS_TRAIN', 'PENDIKOM_TRAIN',
               'BENCHMARK_FULL', 'BENCHMARK_RESAMPLING'],
    'lms_stochastic_runs': 15,
    'holdout_stochastic_runs': 15,
    'benchmark_stochastic_runs': 10,
    'resample_runs': 30,
    'save_semantic_partitions': True,
    'reuse_previous_metrics': True,
    'force': False,
}


## Sumber BK-Modes dan kebijakan implementasi

Thapaliya & Zhuang, *Fast Clustering of Categorical Big Data*,
[arXiv:2502.07081v2](https://arxiv.org/abs/2502.07081v2), Algorithms 8–9,
15 Februari 2025. Ini preprint; kode berikut adalah implementasi ulang
Python, bukan kode C resmi penulis.

Biseksi memilih leaf dengan jumlah SMD terbesar, menggunakan mode leaf
dan objek terjauh sebagai dua pusat awal, dan mengembalikan K mode awal.
Assignment Two-Modes yang seri menuju child kedua sesuai Algorithm 9.
Pilihan yang tidak dirinci paper dicatat dalam `IMPLEMENTATION_POLICY`.
Semua kerja biseksi masuk waktu inisialisasi; optimizer akhir berasal
dari `LIB/care_km.py` dan tidak diubah.

CCI_DOFD belum dimasukkan karena protokol fuzzy/berbobotnya memerlukan
evaluasi kesesuaian terpisah. API initializer BK-Modes dan sumbernya tetap
dapat dimuat dari sel bertanda `bk_modes_library` di bawah ini.


In [3]:
BK_MODES_SOURCE = r'''"""Python reimplementation of Algorithms 8–9, arXiv:2502.07081v2.

This module returns initial modes only. The final K-Modes optimization is
delegated to the unchanged optimizer of the host experiment pipeline.
Paper-unspecified tie/empty/cap policies are explicit in IMPLEMENTATION_POLICY.
"""
import numpy as np

BK_IMPLEMENTATION_VERSION = 'bk-modes-2025-python-1.0.0'
PAPER_URL = 'https://arxiv.org/abs/2502.07081v2'
IMPLEMENTATION_POLICY = {
    'source': PAPER_URL,
    'publication_status': 'arXiv preprint, v2, 15 February 2025',
    'implementation': 'Python reimplementation; not author-supplied C code',
    'cluster_selection': 'largest within-leaf sum of SMD; first-created leaf on a tie',
    'two_modes_initial_centers': 'mode of selected leaf and first farthest source row',
    'two_modes_assignment_tie': 'second child, as in Algorithm 9',
    'initial_categorical_mode_tie': 'lexicographically smallest string',
    'updated_categorical_mode_tie': 'retain previous category if modal; otherwise lexicographic',
    'farthest_row_tie': 'first in the preserved source-row order',
    'duplicate_rows': 'retained with their original frequencies',
    'identical_leaf': 'not bisected; select another splittable leaf',
    'empty_child': 'move first farthest row from a non-singleton donor, then update modes; logged',
    'inner_stopping': 'child memberships unchanged; explicit iteration cap raises an error',
    'final_kmodes': 'host care_km.kmodes_fit, unchanged; host tie and empty-cluster rules',
    'timing_scope': 'all leaf selection, bisection, and mode construction belong to initialization',
    'number_of_centers': 'k leaf modes; coincident leaf modes are reported, not silently deduplicated',
    'determinism': 'fixed implementation, string categories, input order, and parameters',
}


def _validate_X(X):
    X = np.asarray(X, dtype=object)
    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
        raise ValueError('X harus berupa matriks kategorikal dengan baris dan fitur yang tidak kosong.')
    if any(not isinstance(value, (str, np.str_)) for value in X.flat):
        raise TypeError('Kategori X harus berupa string yang sudah disiapkan secara konsisten.')
    if any(value == '' for value in X.flat):
        raise ValueError('Kategori kosong harus ditangani sebelum inisialisasi.')
    return X


def _smd(X, centers):
    """An n-by-k SMD matrix; avoid an n-by-k-by-d temporary."""
    result = np.zeros((len(X), len(centers)), dtype=np.int64)
    for feature in range(X.shape[1]):
        result += X[:, feature, None] != centers[None, :, feature]
    return result


def _mode(X, previous=None):
    center = []
    for feature in range(X.shape[1]):
        values, counts = np.unique(X[:, feature], return_counts=True)
        winners = values[counts == counts.max()]
        old = previous[feature] if previous is not None else None
        center.append(old if old in winners else winners[0])
    return np.asarray(center, dtype=object)


def _assign_two(X, centers):
    distances = _smd(X, centers)
    # The strict comparison and second-child tie implement Algorithm 9.
    labels = np.where(distances[:, 0] < distances[:, 1], 0, 1)
    return labels.astype(np.int64), distances


def _two_modes(X, max_iter):
    first = _mode(X)
    distances = _smd(X, first[None, :])[:, 0]
    farthest = int(np.argmax(distances))
    if int(distances[farthest]) == 0:
        raise ValueError('Leaf dengan satu vektor unik tidak dapat dibelah.')
    centers = np.asarray([first, X[farthest].copy()], dtype=object)
    previous_labels = None
    repairs = 0
    history = []
    for iteration in range(1, max_iter + 1):
        labels, matrix = _assign_two(X, centers)
        counts = np.bincount(labels, minlength=2)
        for empty in np.flatnonzero(counts == 0):
            assigned_distance = matrix[np.arange(len(X)), labels]
            candidates = [int(i) for i in np.argsort(-assigned_distance, kind='stable')
                          if counts[labels[i]] > 1]
            if not candidates:
                raise RuntimeError('Tidak ada donor untuk memperbaiki child kosong.')
            selected = candidates[0]
            counts[labels[selected]] -= 1
            labels[selected] = int(empty)
            counts[empty] += 1
            repairs += 1
        centers = np.asarray([_mode(X[labels == child], centers[child]) for child in range(2)], dtype=object)
        objective = int(_smd(X, centers)[np.arange(len(X)), labels].sum())
        history.append(objective)
        if len(history) > 1 and history[-1] > history[-2]:
            raise RuntimeError('Objective Two-Modes meningkat pada pembaruan.')
        if previous_labels is not None and np.array_equal(labels, previous_labels):
            return {'labels': labels, 'centers': centers, 'iterations': iteration,
                    'empty_repairs': repairs, 'objective_history': history,
                    'initial_farthest_local_row': farthest}
        previous_labels = labels.copy()
    raise RuntimeError(f'Two-Modes belum konvergen setelah {max_iter} iterasi; hasil parsial tidak dipakai.')


def bk_modes_init(X, k, max_iter=100, return_details=False):
    """Return k initial leaf modes without running final global K-Modes.

    ``return_details=True`` also returns the leaf memberships and split trace.
    Categories must be explicit strings, as in the host pipeline datasets.
    The source paper leaves several implementation choices unspecified;
    those choices are recorded in IMPLEMENTATION_POLICY rather than hidden.
    """
    X = _validate_X(X)
    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 1:
        raise ValueError('k harus berupa bilangan bulat positif.')
    if isinstance(max_iter, bool) or not isinstance(max_iter, (int, np.integer)) or max_iter < 1:
        raise ValueError('max_iter harus berupa bilangan bulat positif.')
    k, max_iter = int(k), int(max_iter)
    unique_rows = len({tuple(row) for row in X})
    if k > unique_rows:
        raise ValueError(f'k={k} melebihi jumlah vektor kategorikal unik ({unique_rows}).')

    def node(identifier, indices, center=None):
        center = _mode(X[indices]) if center is None else np.asarray(center, dtype=object).copy()
        cost = int(_smd(X[indices], center[None, :]).sum())
        return {'node_id': int(identifier), 'indices': indices, 'center': center, 'cost': cost}

    root = node(0, np.arange(len(X), dtype=np.int64))
    leaves = {0: root}
    next_id = 1
    trace = []
    for step in range(1, k):
        splittable = [leaf for leaf in leaves.values() if leaf['cost'] > 0]
        if not splittable:
            raise RuntimeError('Jumlah leaf yang diminta tidak tercapai.')
        parent = min(splittable, key=lambda leaf: (-leaf['cost'], leaf['node_id']))
        local = _two_modes(X[parent['indices']], max_iter)
        children = []
        for child in range(2):
            indices = parent['indices'][local['labels'] == child]
            if not len(indices):
                raise RuntimeError('Two-Modes menghasilkan child kosong.')
            children.append(node(next_id + child, indices, local['centers'][child]))
        after_cost = sum(child['cost'] for child in children)
        if after_cost > parent['cost']:
            raise RuntimeError('Biseksi meningkatkan jumlah SMD dalam leaf.')
        trace.append({'step': step, 'parent_node': parent['node_id'],
                      'parent_rows': int(len(parent['indices'])), 'parent_cost': parent['cost'],
                      'child_nodes': [child['node_id'] for child in children],
                      'child_rows': [int(len(child['indices'])) for child in children],
                      'child_costs': [child['cost'] for child in children],
                      'parent_mode': parent['center'].tolist(),
                      'farthest_source_row': int(parent['indices'][local['initial_farthest_local_row']]),
                      'two_modes_iterations': local['iterations'],
                      'empty_repairs': local['empty_repairs'],
                      'objective_history': local['objective_history']})
        del leaves[parent['node_id']]
        for child in children:
            leaves[child['node_id']] = child
        next_id += 2
    ordered = [leaves[key] for key in sorted(leaves)]
    modes = np.asarray([leaf['center'] for leaf in ordered], dtype=object)
    labels = np.full(len(X), -1, dtype=np.int64)
    for label, leaf in enumerate(ordered):
        labels[leaf['indices']] = label
    if np.any(labels < 0) or len(ordered) != k:
        raise RuntimeError('Keanggotaan leaf belum lengkap.')
    details = {'modes': modes, 'leaf_labels': labels, 'split_trace': trace,
               'initial_global_mode_cost': root['cost'],
               'leaf_partition_cost': sum(leaf['cost'] for leaf in ordered),
               'initial_assignment_cost': int(_smd(X, modes).min(axis=1).sum()),
               'unique_initial_modes': len({tuple(row) for row in modes}),
               'total_two_modes_iterations': sum(row['two_modes_iterations'] for row in trace),
               'empty_repairs': sum(row['empty_repairs'] for row in trace),
               'version': BK_IMPLEMENTATION_VERSION}
    return details if return_details else modes
'''

_bk25 = types.ModuleType('_bk25')
exec(compile(BK_MODES_SOURCE, '_bk25', 'exec'), _bk25.__dict__)


## Verifikasi awal

Verifikasi meliputi 55 pemeriksaan: aturan SMD/tie, baris duplikat, input
tidak valid, determinisme, dan 40 kasus dengan implementasi reference
independen. Hasil verifikasi juga diperiksa berdasarkan isi saat digunakan
kembali. Pemeriksaan ini dijalankan sebelum eksperimen besar dimulai.


In [4]:
VERIFICATION_SOURCE = r'''from collections import Counter
import numpy as np
import pandas as pd

def _reference(X, k, max_iter=100):
    """Slow list-based reading of Algorithms 8–9, independent of vectorized helpers."""
    records = [tuple(row) for row in X]
    def distance(a, b):
        return sum(x != y for x, y in zip(a, b))
    def mode(indices, old=None):
        answer = []
        for feature in range(len(records[0])):
            frequencies = Counter(records[i][feature] for i in indices)
            winners = sorted(value for value, count in frequencies.items() if count == max(frequencies.values()))
            answer.append(old[feature] if old is not None and old[feature] in winners else winners[0])
        return tuple(answer)
    def cost(indices, center):
        return sum(distance(records[i], center) for i in indices)
    members = list(range(len(records)))
    center = mode(members)
    leaves = {0: (members, center, cost(members, center))}
    next_id = 1
    parents = []
    for _ in range(k - 1):
        parent_id = min((key for key in leaves if leaves[key][2] > 0),
                        key=lambda key: (-leaves[key][2], key))
        parents.append(parent_id)
        members, center, _ = leaves.pop(parent_id)
        centers = [mode(members)]
        farthest = max(members, key=lambda i: distance(records[i], centers[0]))
        centers.append(records[farthest])
        previous = None
        for iteration in range(max_iter):
            assignments = [0 if distance(records[i], centers[0]) < distance(records[i], centers[1]) else 1 for i in members]
            counts = Counter(assignments)
            for empty in (child for child in (0, 1) if counts[child] == 0):
                order = sorted(range(len(members)),
                               key=lambda pos: (-distance(records[members[pos]], centers[assignments[pos]]), pos))
                pos = next(pos for pos in order if counts[assignments[pos]] > 1)
                counts[assignments[pos]] -= 1
                assignments[pos] = empty
                counts[empty] += 1
            children = [[i for i, child in zip(members, assignments) if child == h] for h in (0, 1)]
            centers = [mode(children[h], centers[h]) for h in (0, 1)]
            if previous is not None and assignments == previous:
                break
            previous = assignments[:]
        else:
            raise RuntimeError('Reference Two-Modes did not converge.')
        for h in (0, 1):
            leaves[next_id + h] = (children[h], centers[h], cost(children[h], centers[h]))
        next_id += 2
    labels = [-1] * len(records)
    centers = []
    for label, node_id in enumerate(sorted(leaves)):
        members, center, _ = leaves[node_id]
        centers.append(center)
        for i in members:
            labels[i] = label
    return {'modes': np.asarray(centers, dtype=object), 'labels': np.asarray(labels), 'parents': parents}

def small_case_verification(bk):
    checks = []
    def check(name, action):
        action()
        checks.append({'pemeriksaan': name, 'status': 'LULUS'})
    def require(condition, message='Hasil tidak sesuai kontrak.'):
        if not condition:
            raise AssertionError(message)
    def rejected(action, error_type):
        try:
            action()
        except error_type:
            return
        raise AssertionError('Input tidak valid tidak ditolak.')

    X = np.asarray([['a', 'a'], ['a', 'b'], ['b', 'a'], ['b', 'b']], dtype=object)
    check('SMD sama dengan hitungan mismatch langsung', lambda: require(
        np.array_equal(bk._smd(X, X), np.asarray([[sum(a != b for a, b in zip(x, y)) for y in X] for x in X]))))
    check('Assignment Two-Modes yang seri menuju child kedua (Algorithm 9)', lambda: require(
        bk._assign_two(np.asarray([['a', 'b']], dtype=object), np.asarray([['a', 'a'], ['b', 'b']], dtype=object))[0].tolist() == [1]))
    check('Mode awal dengan kategori seri memakai urutan leksikografis', lambda: require(
        bk._mode(np.asarray([['b'], ['a']], dtype=object)).tolist() == ['a']))
    check('Mode pembaruan mempertahankan kategori modal sebelumnya', lambda: require(
        bk._mode(np.asarray([['b'], ['a']], dtype=object), np.asarray(['b'], dtype=object)).tolist() == ['b']))
    check('k=1 menghasilkan mode global tanpa biseksi', lambda: require(
        bk.bk_modes_init(X, 1, return_details=True)['split_trace'] == [] and
        bk.bk_modes_init(X, 1).tolist() == [['a', 'a']]))
    repeated = np.asarray([['a'], ['a'], ['a'], ['b']], dtype=object)
    check('Frekuensi baris duplikat dipertahankan', lambda: require(bk.bk_modes_init(repeated, 1).tolist() == [['a']]))
    check('k melebihi vektor unik ditolak', lambda: rejected(lambda: bk.bk_modes_init(repeated, 3), ValueError))
    check('Matriks kosong ditolak', lambda: rejected(lambda: bk.bk_modes_init(np.empty((0, 2), dtype=object), 1), ValueError))
    check('Kategori kosong ditolak', lambda: rejected(lambda: bk.bk_modes_init([['']], 1), ValueError))
    check('Kategori non-string harus dikonversi eksplisit', lambda: rejected(lambda: bk.bk_modes_init([[1]], 1), TypeError))
    check('k boolean ditolak', lambda: rejected(lambda: bk.bk_modes_init(X, True), ValueError))
    check('Batas iterasi tidak mengembalikan solusi parsial diam-diam', lambda: rejected(
        lambda: bk.bk_modes_init(X, 2, max_iter=1), RuntimeError))
    check('Farthest tie memilih baris sumber pertama', lambda: require(
        bk.bk_modes_init(np.asarray([['a', 'a'], ['a', 'b'], ['b', 'a']], dtype=object), 2,
                         return_details=True)['split_trace'][0]['farthest_source_row'] == 1))
    original = X.copy()
    a = bk.bk_modes_init(X, 3, return_details=True)
    b = bk.bk_modes_init(X, 3, return_details=True)
    check('Eksekusi ulang deterministik pada input yang sama', lambda: require(
        np.array_equal(a['modes'], b['modes']) and np.array_equal(a['leaf_labels'], b['leaf_labels']) and a['split_trace'] == b['split_trace']))
    check('Input tidak dimutasi', lambda: require(np.array_equal(X, original)))

    rng = np.random.default_rng(250207081)
    for case in range(40):
        n, d = int(rng.integers(5, 22)), int(rng.integers(1, 6))
        data = rng.choice(np.asarray(['a', 'b', 'c'], dtype=object), size=(n, d)).astype(object)
        k = min(1 + case % 5, len({tuple(row) for row in data}))
        actual = bk.bk_modes_init(data, k, return_details=True)
        reference = _reference(data, k)
        require(np.array_equal(actual['modes'], reference['modes']), 'Mode berbeda dari reference independen.')
        require(np.array_equal(actual['leaf_labels'], reference['labels']), 'Leaf berbeda dari reference independen.')
        require([row['parent_node'] for row in actual['split_trace']] == reference['parents'], 'Urutan leaf berbeda.')
        require(len(actual['split_trace']) == k - 1)
        require(actual['initial_assignment_cost'] <= actual['initial_global_mode_cost'])
        checks.append({'pemeriksaan': f'Reference independen kasus {case + 1:02d}, n={n}, d={d}, k={k}', 'status': 'LULUS'})
    return pd.DataFrame(checks)
'''

_verify25 = types.ModuleType('_verify25')
exec(compile(VERIFICATION_SOURCE, '_verify25', 'exec'), _verify25.__dict__)


## Perbandingan terintegrasi dan bank partisi

Satu daftar delapan metode digunakan di seluruh kelompok. Per-run artifact
memisahkan scope full-data, train, dan resampling. Mode full LMS tidak
dipakai sebagai pengganti mode train pada holdout. Assignment test memakai
mode train yang sudah dibekukan.

Keluaran lama dari 01–24 dibaca untuk penggunaan kembali yang terverifikasi.
Jika versi paket ilmiah, input, kode, atau indeks sumber berbeda, hasil yang
terdampak dihitung kembali. Ringkasan dan bank partisi lama tidak ditimpa.

Catatan waktu setiap run dipertahankan dengan asalnya. Notebook tidak
menyimpulkan keunggulan runtime dari campuran timing tersimpan dan timing
baru; pengukuran runtime komparatif terkontrol dilakukan pada eksperimen
runtime tersendiri. Kegagalan algoritmik dicatat, bukan diubah menjadi sukses.


In [5]:
COMPARISON_SOURCE = r'''"""Eight-initializer experiments, shared optimizer, persistent per-run artifacts."""
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import importlib.metadata
import io
import itertools
import json
import os
import platform
import re
import sys
import time
import types
import uuid
import numpy as np
import pandas as pd

STAGE = '25_Perbandingan_Initializer_Terintegrasi'
OLD_STAGE = '25_Verifikasi_Baseline_BK_Modes_2025'
METHODS = ['Random', 'Huang', 'Cao', 'Bai', 'K-Modes++', 'HR-2023', 'CARE-KM', 'BK-Modes']
STOCHASTIC = {'Random', 'Huang', 'K-Modes++', 'HR-2023'}
SCOPES = ['LMS_FULL', 'LMS_RESAMPLING', 'LMS_TRAIN', 'PENDIKOM_TRAIN',
          'BENCHMARK_FULL', 'BENCHMARK_RESAMPLING']
METRICS = ['initial_cost', 'final_cost', 'dbi', 'si', 'iterations']
DEFAULT_CONFIG = {'scopes': SCOPES, 'lms_stochastic_runs': 15, 'holdout_stochastic_runs': 15,
                  'benchmark_stochastic_runs': 10, 'resample_runs': 30,
                  'save_semantic_partitions': True, 'reuse_previous_metrics': True,
                  'force': False}
INPUT_LMS = 'OUTPUT/PRAPROSES/01_Praproses_main_LMS/05_dataset_behavioral_categorical_skala.csv'
TRAIN_INPUTS = {
    'LMS_TRAIN': ('LMS', 'user', 4660, 1998),
    'PENDIKOM_TRAIN': ('PENDIKOM', 'userid', 322, 139),
}
OLD_SOURCES = {
    'LMS_FULL': ('07_Eksperimen_LMS_Full', 'OUTPUT/EKSPERIMEN/LMS/07_Eksperimen_LMS_Full/04_full_dataset_raw.csv', 'rep'),
    'LMS_RESAMPLING': ('09_Robustness_LMS_30x80', 'OUTPUT/EKSPERIMEN/LMS/09_Robustness_LMS_30x80/06_robustness_raw.csv', 'rep'),
    'PENDIKOM_TRAIN': ('22_Holdout_LMS_Pendikom', 'OUTPUT/EKSPERIMEN/HOLDOUT/PENDIKOM/70_30_groundtruth_clustering_baseline_raw.csv', 'rep'),
    'BENCHMARK_FULL': ('19_Generalisasi_Benchmark', 'OUTPUT/EKSPERIMEN/GENERALISASI/19_Generalisasi_Benchmark/10_cross_dataset_full_raw.csv', 'run'),
    'BENCHMARK_RESAMPLING': ('19_Generalisasi_Benchmark', 'OUTPUT/EKSPERIMEN/GENERALISASI/19_Generalisasi_Benchmark/10_cross_dataset_resample_raw.csv', 'rep'),
}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                    separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()


def binary_hash(data):
    return hashlib.sha256(data).hexdigest()


def clean(value):
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean(v) for v in value]
    if isinstance(value, np.ndarray):
        return clean(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.bool_):
        return bool(value)
    return value


def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(clean(value), indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temp.replace(path)


def save_csv(path, frame):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temp, index=False, lineterminator='\n')
    temp.replace(path)


def read_checkpoint(path):
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
        return data if isinstance(data, dict) else {}
    except (OSError, UnicodeError, json.JSONDecodeError):
        return {}


class Files:
    """Fresh content hashes for each call; reuse reads only while stat is unchanged."""
    def __init__(self, root):
        self.root, self.cache = Path(root).resolve(), {}

    def read(self, relative):
        path = self.root / relative
        before = path.stat()
        stamp = (before.st_size, before.st_mtime_ns, before.st_ctime_ns)
        if relative in self.cache and self.cache[relative][0] == stamp:
            return self.cache[relative][1]
        data = path.read_bytes()
        after = path.stat()
        if stamp != (after.st_size, after.st_mtime_ns, after.st_ctime_ns):
            raise RuntimeError('Berkas berubah saat dibaca: ' + relative)
        self.cache[relative] = (stamp, data)
        return data

    def sha(self, relative):
        return binary_hash(self.read(relative))


def runtime():
    packages = {}
    for name, module in [('numpy', 'numpy'), ('pandas', 'pandas'), ('scipy', 'scipy'), ('sklearn', 'sklearn')]:
        loaded = sys.modules.get(module)
        packages[name] = getattr(loaded, '__version__', None) or importlib.metadata.version('scikit-learn' if name == 'sklearn' else name)
    return {'python': platform.python_version(), 'packages': packages}


def literals(source, requested):
    result = {}
    for node in ast.parse(source.decode('utf-8-sig')).body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            name = node.targets[0].id
            if name in requested:
                result[name] = ast.literal_eval(node.value)
    if set(result) != set(requested):
        raise RuntimeError('Konfigurasi pustaka belum lengkap: ' + str(set(requested) - set(result)))
    return result


def load_module(source, name, filename):
    module = types.ModuleType(name)
    module.__file__ = str(filename)
    exec(compile(source, str(filename), 'exec'), module.__dict__)
    return module


def load_project(files):
    core = load_module(files.read('LIB/care_km.py'), '_initializer25_core', files.root / 'LIB/care_km.py')
    gd = load_module(files.read('LIB/generalization_data.py'), '_initializer25_datasets', files.root / 'LIB/generalization_data.py')
    constants = literals(files.read('LIB/experiment_lib.py'), ['K', 'MAX_ITER', 'SI_SAMPLE_SIZE', 'FROZEN_CARE_CONFIG'])
    features = literals(files.read('LIB/pipeline_data.py'), ['MAIN_FEATURES', 'PENDIKOM_FEATURES'])
    features['PENDIKOM_USED'] = [c for c in features['PENDIKOM_FEATURES'] if c != 'navigation_skip']
    if constants['K'] != 16 or constants['MAX_ITER'] != 100:
        raise RuntimeError('Protokol utama harus k=16 dan MAX_ITER=100 untuk paket ini.')
    return core, gd, constants, features


def read_frame(files, path):
    return pd.read_csv(io.BytesIO(files.read(path)), dtype=str, keep_default_na=False)


def validate_config(configuration, gd):
    unknown = set(configuration) - set(DEFAULT_CONFIG) - {'benchmark_datasets'}
    if unknown:
        raise ValueError('Nama konfigurasi tidak dikenal: ' + str(sorted(unknown)))
    cfg = dict(DEFAULT_CONFIG)
    cfg.update(configuration)
    cfg['scopes'] = list(cfg['scopes'])
    cfg['benchmark_datasets'] = list(cfg.get('benchmark_datasets', gd.DATASET_ORDER))
    if not cfg['scopes'] or len(set(cfg['scopes'])) != len(cfg['scopes']) or set(cfg['scopes']) - set(SCOPES):
        raise ValueError('Pilihan kelompok eksperimen tidak sesuai daftar.')
    if not cfg['benchmark_datasets'] or len(set(cfg['benchmark_datasets'])) != len(cfg['benchmark_datasets']) or set(cfg['benchmark_datasets']) - set(gd.DATASET_ORDER):
        raise ValueError('Pilihan benchmark tidak sesuai registry.')
    for key in ['lms_stochastic_runs', 'holdout_stochastic_runs', 'benchmark_stochastic_runs', 'resample_runs']:
        if isinstance(cfg[key], bool) or not isinstance(cfg[key], int) or cfg[key] < 1:
            raise ValueError('Jumlah pengulangan tidak valid: ' + key)
    if cfg['resample_runs'] > 30:
        raise ValueError('Paket menyediakan paling banyak 30 indeks resampling.')
    for key in ['save_semantic_partitions', 'reuse_previous_metrics', 'force']:
        if not isinstance(cfg[key], bool):
            raise ValueError('Konfigurasi harus True/False: ' + key)
    return cfg


def planned_jobs(cfg):
    per_repeat = len(METHODS)
    one_off = len(METHODS) - len(STOCHASTIC)
    counts = {
        'LMS_FULL': one_off + len(STOCHASTIC) * cfg['lms_stochastic_runs'],
        'LMS_RESAMPLING': per_repeat * cfg['resample_runs'],
        'LMS_TRAIN': one_off + len(STOCHASTIC) * cfg['holdout_stochastic_runs'],
        'PENDIKOM_TRAIN': one_off + len(STOCHASTIC) * cfg['holdout_stochastic_runs'],
        'BENCHMARK_FULL': len(cfg['benchmark_datasets']) * (one_off + len(STOCHASTIC) * cfg['benchmark_stochastic_runs']),
        'BENCHMARK_RESAMPLING': len(cfg['benchmark_datasets']) * per_repeat * cfg['resample_runs'],
    }
    return {scope: counts[scope] for scope in cfg['scopes']}


def preflight(files, cfg, gd):
    required = ['LIB/care_km.py', 'LIB/experiment_lib.py', 'LIB/pipeline_data.py', 'LIB/generalization_data.py',
                'PYTHON/' + STAGE + '.ipynb']
    if set(cfg['scopes']) & {'LMS_FULL', 'LMS_RESAMPLING'}:
        required.append(INPUT_LMS)
    for scope, (dataset, identity, ntr, nte) in TRAIN_INPUTS.items():
        if scope in cfg['scopes']:
            required += [f'OUTPUT/SPLIT/{dataset}/{part}/categorical.csv' for part in ('TRAIN', 'TEST')]
    if 'LMS_RESAMPLING' in cfg['scopes']:
        required += [f'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{r:02d}.csv' for r in range(cfg['resample_runs'])]
    if set(cfg['scopes']) & {'BENCHMARK_FULL', 'BENCHMARK_RESAMPLING'}:
        for dataset in cfg['benchmark_datasets']:
            required.append('DATA/' + gd.DATASETS[dataset]['file'])
            if dataset == 'chronic_kidney':
                for filename in ('chronic_kidney_disease_full.arff', 'chronic_kidney_disease.arff'):
                    if (files.root / 'DATA' / filename).is_file():
                        required.append('DATA/' + filename)
            if 'BENCHMARK_RESAMPLING' in cfg['scopes']:
                required += [f'OUTPUT/SPLIT/RESAMPLING/BENCHMARK/19_Generalisasi_Benchmark/{dataset}/rep_{r:02d}.csv'
                             for r in range(cfg['resample_runs'])]
    missing = [p for p in required if not (files.root / p).is_file()]
    if missing:
        raise FileNotFoundError('Masukan belum lengkap. Lengkapi dari proyek sebelumnya:\n' + '\n'.join(missing[:20]))
    # Colab autosaves rendered cell outputs during long runs. Notebook source
    # is monitored separately; whole-file output/count changes are not inputs.
    return {p: files.sha(p) for p in required if not p.endswith('.ipynb')}


def notebook_code(files):
    notebook = json.loads(files.read('PYTHON/' + STAGE + '.ipynb').decode('utf-8-sig'))
    code = []
    for cell in notebook['cells']:
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source'])
        if 'initializer25_configuration' in cell.get('metadata', {}).get('tags', []):
            tree = ast.parse(source)
            if (len(tree.body) != 1 or not isinstance(tree.body[0], ast.Assign)
                    or len(tree.body[0].targets) != 1 or not isinstance(tree.body[0].targets[0], ast.Name)
                    or tree.body[0].targets[0].id != 'KONFIGURASI_25'
                    or not isinstance(ast.literal_eval(tree.body[0].value), dict)):
                raise RuntimeError('Sel konfigurasi hanya boleh berisi assignment literal KONFIGURASI_25.')
            continue
        code.append(source)
    return digest(code)


def same_inputs(files, initial, expected_code=None):
    for path, expected in initial.items():
        if files.sha(path) != expected:
            raise RuntimeError('Masukan berubah di tengah eksekusi: ' + path)
    if expected_code is not None and notebook_code(files) != expected_code:
        raise RuntimeError('Kode notebook berubah selama eksekusi; simpan kode tetap lalu jalankan kembali.')


def indices(files, relative, n, size):
    data = pd.read_csv(io.BytesIO(files.read(relative)))
    idx = data['source_row_index'].to_numpy(dtype=np.int64)
    if len(idx) != size or len(np.unique(idx)) != size or np.any(idx < 0) or np.any(idx >= n):
        raise RuntimeError('Indeks matched-resampling tidak valid: ' + relative)
    return idx


def _context(scope, dataset, frame, feature_names, identity, k, sources, rep=None, original_indices=None,
             test_frame=None, y=None, cfg=None):
    X = frame[feature_names].astype(str).to_numpy(dtype=object)
    if len({tuple(row) for row in X}) < k:
        raise RuntimeError(f'{scope}/{dataset}: k melebihi jumlah vektor unik.')
    ids = frame[identity].astype(str).tolist() if identity in frame else [str(i) for i in range(len(frame))]
    if len(set(ids)) != len(ids):
        raise RuntimeError('Identitas peserta/baris tidak unik: ' + scope + '/' + dataset)
    idx = np.arange(len(X), dtype=np.int64) if original_indices is None else np.asarray(original_indices, dtype=np.int64)
    if scope.endswith('RESAMPLING'):
        runs, base_seed, si_seed = 1, (6161600 if scope == 'LMS_RESAMPLING' else 130000) + rep, None
    elif scope == 'BENCHMARK_FULL':
        runs, base_seed, si_seed = cfg['benchmark_stochastic_runs'], 120000, 120999
    elif scope in TRAIN_INPUTS:
        runs, base_seed, si_seed = cfg['holdout_stochastic_runs'], 4161600, None
    else:
        runs, base_seed, si_seed = cfg['lms_stochastic_runs'], 4161600, None
    Xtest = test_frame[feature_names].astype(str).to_numpy(dtype=object) if test_frame is not None else None
    test_ids = test_frame[identity].astype(str).tolist() if test_frame is not None else []
    if set(ids) & set(test_ids):
        raise RuntimeError('Train/test memiliki peserta tumpang tindih.')
    result = {'scope': scope, 'dataset': dataset, 'rep': rep, 'X': X, 'y': None if y is None else np.asarray(y, dtype=str),
              'features': feature_names, 'identity': identity, 'ids': ids, 'indices': idx,
              'test_X': Xtest, 'test_ids': test_ids, 'k': int(k), 'sources': sources,
              'stochastic_runs': runs, 'base_seed': base_seed, 'fixed_si_seed': si_seed,
              'si_sample_size': None if scope == 'PENDIKOM_TRAIN' else 2000,
              'needs_partition': scope in TRAIN_INPUTS or (scope.startswith('LMS_') and cfg['save_semantic_partitions'])}
    result['matrix_sha256'] = digest({'features': feature_names, 'rows': X.tolist(), 'ids': ids,
                                      'indices': idx.tolist(), 'test_rows': Xtest.tolist() if Xtest is not None else None,
                                      'test_ids': test_ids, 'classes': result['y'].tolist() if result['y'] is not None else None})
    X.flags.writeable = False
    return result


def contexts(files, cfg, gd, constants, features):
    scopes = set(cfg['scopes'])
    if scopes & {'LMS_FULL', 'LMS_RESAMPLING'}:
        frame = read_frame(files, INPUT_LMS)
        if len(frame) != 6658 or frame['user'].duplicated().any() or list(frame.columns) != ['user'] + features['MAIN_FEATURES']:
            raise RuntimeError('Data LMS full harus 6.658 peserta dengan urutan 13 fitur yang ditetapkan.')
        if 'LMS_FULL' in scopes:
            yield _context('LMS_FULL', 'LMS', frame, features['MAIN_FEATURES'], 'user', 16, [INPUT_LMS], cfg=cfg)
        if 'LMS_RESAMPLING' in scopes:
            for rep in range(cfg['resample_runs']):
                path = f'OUTPUT/SPLIT/RESAMPLING/LMS/09_Robustness_LMS_30x80/rep_{rep:02d}.csv'
                idx = indices(files, path, len(frame), int(len(frame) * .8))
                yield _context('LMS_RESAMPLING', 'LMS', frame.iloc[idx], features['MAIN_FEATURES'], 'user', 16,
                               [INPUT_LMS, path], rep=rep, original_indices=idx, cfg=cfg)
    for scope, (dataset, identity, ntr, nte) in TRAIN_INPUTS.items():
        if scope not in scopes:
            continue
        train_path, test_path = [f'OUTPUT/SPLIT/{dataset}/{part}/categorical.csv' for part in ('TRAIN', 'TEST')]
        train, test = read_frame(files, train_path), read_frame(files, test_path)
        if len(train) != ntr or len(test) != nte:
            raise RuntimeError('Jumlah peserta holdout tidak sesuai protokol: ' + dataset)
        names = features['MAIN_FEATURES'] if dataset == 'LMS' else features['PENDIKOM_USED']
        if 'split' in train and (not train['split'].str.lower().eq('train').all() or not test['split'].str.lower().eq('test').all()):
            raise RuntimeError('Identitas bagian train/test tidak sesuai.')
        yield _context(scope, dataset, train, names, identity, 16, [train_path, test_path], test_frame=test, cfg=cfg)
    if scopes & {'BENCHMARK_FULL', 'BENCHMARK_RESAMPLING'}:
        for dataset in cfg['benchmark_datasets']:
            spec = gd.DATASETS[dataset]
            X, y = gd.load_dataset(files.root / 'DATA', dataset)
            if (len(X), X.shape[1], int(pd.Series(y).nunique())) != (spec['reference_rows'], spec['reference_features'], spec['reference_k']):
                raise RuntimeError('Skema benchmark tidak sesuai registry: ' + dataset)
            frame = X.copy()
            frame['_row_id'] = np.arange(len(frame)).astype(str)
            sources = ['DATA/' + spec['file']]
            if dataset == 'chronic_kidney':
                for filename in ('chronic_kidney_disease_full.arff', 'chronic_kidney_disease.arff'):
                    if (files.root / 'DATA' / filename).is_file():
                        sources.append('DATA/' + filename)
            names, k = list(X.columns), int(pd.Series(y).nunique())
            if 'BENCHMARK_FULL' in scopes:
                yield _context('BENCHMARK_FULL', dataset, frame, names, '_row_id', k, sources, y=y, cfg=cfg)
            if 'BENCHMARK_RESAMPLING' in scopes:
                for rep in range(cfg['resample_runs']):
                    path = f'OUTPUT/SPLIT/RESAMPLING/BENCHMARK/19_Generalisasi_Benchmark/{dataset}/rep_{rep:02d}.csv'
                    idx = indices(files, path, len(frame), int(round(len(frame) * .8)))
                    yield _context('BENCHMARK_RESAMPLING', dataset, frame.iloc[idx], names, '_row_id', k, sources + [path],
                                   rep=rep, original_indices=idx, y=np.asarray(y)[idx], cfg=cfg)


def job_parameters(context, method, run, constants):
    seed = context['base_seed'] + run
    return {'scope': context['scope'], 'dataset': context['dataset'], 'rep': context['rep'], 'run': run,
            'method': method, 'seed': seed,
            'si_seed': context['fixed_si_seed'] if context['fixed_si_seed'] is not None else seed,
            'si_sample_size': context['si_sample_size'], 'k': context['k'],
            'max_iter': constants['MAX_ITER'], 'care_config': constants['FROZEN_CARE_CONFIG'],
            'matrix_sha256': context['matrix_sha256'], 'needs_partition': context['needs_partition']}


def _artifact_inventory(root, paths, files):
    return {str(path.relative_to(root)).replace('\\', '/'): files.sha(str(path.relative_to(root)).replace('\\', '/')) for path in paths}


class Legacy:
    """Reuse only output rows with matching recorded content and relevant protocol."""
    def __init__(self, files, current_runtime, scientific_hashes):
        self.files, self.runtime, self.scientific_hashes = files, current_runtime, scientific_hashes
        path = files.root / 'OUTPUT/KONTROL/status_pipeline.json'
        try:
            records = json.loads(path.read_text(encoding='utf-8'))
            self.records = {r['notebook']: r for r in records if isinstance(r, dict) and 'notebook' in r} if isinstance(records, list) else {}
        except (OSError, UnicodeError, json.JSONDecodeError):
            self.records = {}
        self.frames, self.reasons = {}, {}

    def get(self, context, parameters):
        scope, method = context['scope'], parameters['method']
        if method == 'BK-Modes' or context['needs_partition'] or scope not in OLD_SOURCES:
            return None
        stem, relative, trial = OLD_SOURCES[scope]
        record = self.records.get(stem, {})
        def reject(reason):
            self.reasons[scope + '/' + context['dataset']] = reason
            return None
        if record.get('status') != 'selesai' or not record.get('output_manifest'):
            return reject('catatan isi keluaran sebelumnya belum lengkap')
        state = record.get('input_manifest', {})
        old_runtime = state.get('runtime', {})
        old_packages = old_runtime.get('packages', {})
        if any(old_packages.get(k) != v for k, v in self.runtime['packages'].items()) or old_runtime.get('python') != self.runtime['python']:
            return reject('versi Python/paket ilmiah berbeda')
        libraries = state.get('libraries', {})
        for filename, checksum in self.scientific_hashes.items():
            if libraries.get(filename, {}).get('sha256') != checksum:
                return reject('isi pustaka ilmiah berubah')
        recorded_output = record['output_manifest'].get(relative, {})
        if not (self.files.root / relative).is_file() or self.files.sha(relative) != recorded_output.get('sha256'):
            return reject('isi keluaran lama tidak sesuai catatan')
        notebook = 'PYTHON/' + stem + '.ipynb'
        if not (self.files.root / notebook).is_file():
            return reject('skrip asal tidak tersedia')
        old_nb = json.loads(self.files.read(notebook).decode('utf-8-sig'))
        old_code = [''.join(c['source']) for c in old_nb['cells'] if c['cell_type'] == 'code']
        if digest(old_code) != state.get('notebooks', {}).get(notebook):
            return reject('skrip asal berubah')
        known = {}
        known.update(state.get('data', {})); known.update(state.get('preprocessed', {}))
        known.update(record.get('output_manifest', {}))
        for dependency in state.get('dependencies', {}).values():
            known.update(dependency)
        for source in context['sources']:
            if known.get(source, {}).get('sha256') != self.files.sha(source):
                return reject('isi data atau indeks sumber berubah/tidak tercatat')
        if relative not in self.frames:
            self.frames[relative] = pd.read_csv(io.BytesIO(self.files.read(relative)), float_precision='round_trip')
        frame = self.frames[relative]
        mask = frame['method'].eq(method) & pd.to_numeric(frame[trial], errors='coerce').eq(context['rep'] if context['rep'] is not None else parameters['run'])
        if scope.startswith('BENCHMARK'):
            mask &= frame['dataset'].eq(context['dataset'])
        matched = frame.loc[mask]
        if len(matched) != 1:
            return reject('baris hasil tidak unik atau belum tersedia')
        row = matched.iloc[0].to_dict()
        if int(row.get('k', -1)) != parameters['k'] or int(row.get('seed', -1)) != parameters['seed']:
            return reject('k/seed tidak sesuai protokol')
        if context['rep'] is not None:
            sample_hash = binary_hash(np.asarray(context['indices'], dtype=np.int64).tobytes())[:16]
            if row.get('sample_hash') != sample_hash:
                return reject('anggota matched-resample berbeda')
        if any(not np.isfinite(float(row.get(field, np.nan))) for field in METRICS):
            return reject('metrik lama tidak valid')
        evidence = {'file': relative, 'sha256': self.files.sha(relative), 'stage': stem,
                    'source_notebook': notebook, 'source_code_sha256': digest(old_code),
                    'source_runtime': old_runtime}
        return clean(row), evidence


def fit_job(core, bk, context, parameters):
    X, k = context['X'], parameters['k']
    t0 = time.perf_counter()
    if parameters['method'] == 'BK-Modes':
        detail = bk.bk_modes_init(X, k, max_iter=parameters['max_iter'], return_details=True)
        init = detail['modes']
    else:
        detail = None
        init = core.init_by_name(parameters['method'], X, k, seed=parameters['seed'], care_config=parameters['care_config'])
    t1 = time.perf_counter()
    fit = core.kmodes_fit(X, init, max_iter=parameters['max_iter'])
    t2 = time.perf_counter()
    flags = []
    counts = np.bincount(fit['labels'], minlength=k)
    if (counts == 0).any(): flags.append('empty_final_cluster')
    initial_cost = float(core.initial_cost(X, init))
    if fit['cost'] > initial_cost + 1e-12: flags.append('objective_worsened')
    if int(fit['iterations']) < 1: flags.append('nonpositive_iterations')
    try:
        metrics = core.evaluate_clustering(X, context['y'], fit, init,
                                           si_sample_size=parameters['si_sample_size'], si_seed=parameters['si_seed'])
    except ValueError as error:
        metrics = {'initial_cost': initial_cost, 'final_cost': float(fit['cost']),
                   'iterations': int(fit['iterations']), 'dbi': None, 'si': None}
        flags.append('metric_error: ' + str(error))
    for field in METRICS:
        if metrics.get(field) is None or not np.isfinite(float(metrics[field])):
            flags.append('nonfinite_' + field)
    row = {**metrics, 'init_sec': t1-t0, 'cluster_sec': t2-t1, 'total_sec': t2-t0,
           'valid': not flags, 'failure_flags': '; '.join(flags), 'origin': 'COMPUTED',
           'runtime_comparable': False}
    return clean(row), init, fit, detail


def store_partition(folder, context, parameters, init, fit, detail, core):
    paths = []
    for filename, array in [('initial_modes.csv', init), ('final_modes.csv', fit['modes'])]:
        frame = pd.DataFrame(array, columns=context['features'])
        frame.insert(0, 'cluster_id', np.arange(parameters['k']))
        path = folder / filename; save_csv(path, frame); paths.append(path)
    assignments = pd.DataFrame({'source_row_index': context['indices'], 'identity': context['ids'], 'cluster_id': fit['labels']})
    path = folder / 'train_assignments.csv'; save_csv(path, assignments); paths.append(path)
    if context['test_X'] is not None:
        labels, distance = core.assign_nearest(context['test_X'], fit['modes'])
        path = folder / 'test_assignments.csv'
        save_csv(path, pd.DataFrame({'identity': context['test_ids'], 'cluster_id': labels, 'distance': distance}))
        paths.append(path)
    if detail is not None:
        path = folder / 'bisection_trace.json'; save_json(path, detail['split_trace']); paths.append(path)
    return paths


def execute_job(root, output, control, context, method, run, constants, core, bk,
                code_state, current_runtime, files, legacy, cfg):
    parameters = job_parameters(context, method, run, constants)
    signature = digest({'parameters': parameters, 'code': code_state, 'runtime': current_runtime})
    identity = {key: parameters[key] for key in ('scope', 'dataset', 'rep', 'run', 'method')}
    key = digest(identity)[:24]
    status_file = control / 'JOBS' / (key + '.json')
    if status_file.is_file() and not cfg['force']:
        previous = read_checkpoint(status_file)
        checksum = previous.pop('record_sha256', '')
        try:
            valid_record = checksum == digest(previous)
        except (TypeError, ValueError):
            valid_record = False
        good_outputs = valid_record and all((root / path).is_file() and files.sha(path) == checksum for path, checksum in previous.get('outputs', {}).items())
        evidence = previous.get('legacy_evidence')
        good_evidence = not evidence or ((root / evidence['file']).is_file() and files.sha(evidence['file']) == evidence['sha256'])
        if valid_record and previous.get('status') == 'selesai' and previous.get('signature') == signature and good_outputs and good_evidence:
            return previous['row'], 'CACHE', previous
    reused = legacy.get(context, parameters) if cfg['reuse_previous_metrics'] and not cfg['force'] else None
    folder = output / 'BANK' / context['scope'] / context['dataset'] / ('r' + str(context['rep'] if context['rep'] is not None else run).zfill(2)) / re.sub(r'[^A-Za-z0-9]+', '_', method.replace('++', '_PP')).strip('_')
    paths, evidence = [], None
    if reused is not None:
        row, evidence = reused
        row.update(valid=True, failure_flags='', origin='REUSED_VERIFIED', runtime_comparable=False)
        disposition = 'REUSE'
    else:
        row, init, fit, detail = fit_job(core, bk, context, parameters)
        # Retain all newly computed partitions, including benchmark runs.
        paths = store_partition(folder, context, parameters, init, fit, detail, core)
        disposition = 'FIT'
    row.update(scope=context['scope'], dataset=context['dataset'], method=method, run=run,
               rep=context['rep'], seed=parameters['seed'], si_seed=parameters['si_seed'],
               k=parameters['k'], sample_n=len(context['X']), matrix_sha256=context['matrix_sha256'],
               sample_hash=binary_hash(np.asarray(context['indices'], dtype=np.int64).tobytes())[:16],
               partition_available=bool(paths), features=len(context['features']))
    record = {'schema': 1, 'status': 'selesai', 'signature': signature, 'identity': identity,
              'parameters': parameters, 'row': clean(row), 'outputs': _artifact_inventory(root, paths, files),
              'legacy_evidence': evidence, 'finished_utc': datetime.now(timezone.utc).isoformat()}
    record['record_sha256'] = digest(record)
    save_json(status_file, record)
    return record['row'], disposition, record


def summarize(raw):
    rows = []
    for (scope, dataset, method), group in raw.groupby(['scope', 'dataset', 'method'], sort=False):
        valid = group.loc[group['valid'].eq(True)]
        row = {'scope': scope, 'dataset': dataset, 'method': method, 'n_runs': len(group),
               'n_valid': len(valid), 'n_failures': len(group)-len(valid),
               'partition_runs': int(group['partition_available'].eq(True).sum())}
        for metric in METRICS:
            values = pd.to_numeric(valid[metric], errors='coerce')
            row[metric + '_mean'], row[metric + '_sd'] = values.mean(), values.std(ddof=1)
        rows.append(row)
    return pd.DataFrame(rows)


def exact_signed_rank(differences, zero_tolerance=1e-12):
    differences = np.asarray(differences, dtype=float)
    if not np.isfinite(differences).all():
        raise ValueError('Paired differences mengandung nilai tidak valid.')
    nonzero = np.abs(differences) > zero_tolerance
    values = differences[nonzero]
    if not len(values):
        return {'statistic': 0.0, 'p_value': 1.0, 'rank_biserial': 0.0, 'n_nonzero': 0,
                'n_zero': len(differences), 'absolute_ranks': [], 'positive_rank_sum': 0.0, 'negative_rank_sum': 0.0}
    ranks = pd.Series(np.abs(values)).rank(method='average').to_numpy(float)
    positive, negative = float(ranks[values > 0].sum()), float(ranks[values < 0].sum())
    observed = min(positive, negative)
    count = 0
    for signs in itertools.product((0, 1), repeat=len(values)):
        pos = float(np.dot(signs, ranks))
        count += min(pos, ranks.sum()-pos) <= observed + 1e-12
    return {'statistic': observed, 'p_value': count / (2 ** len(values)),
            'rank_biserial': (positive-negative)/(positive+negative), 'n_nonzero': len(values),
            'n_zero': int((~nonzero).sum()), 'absolute_ranks': ranks.tolist(),
            'positive_rank_sum': positive, 'negative_rank_sum': negative}


def holm(values):
    order = np.argsort(values, kind='stable')
    answer, running = np.zeros(len(values)), 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(values)-rank)*values[index])
        answer[index] = min(1.0, running)
    return answer


def benchmark_statistics(summary, cfg, gd, output):
    from scipy.stats import friedmanchisquare
    ranks, friedman, pairwise, audit = [], [], [], []
    notes = []
    for scope in ['BENCHMARK_FULL', 'BENCHMARK_RESAMPLING']:
        frame = summary.loc[summary['scope'].eq(scope)]
        if frame.empty:
            notes.append(scope + ': belum dijalankan')
            continue
        if scope == 'BENCHMARK_RESAMPLING' and cfg['resample_runs'] != 30:
            notes.append(scope + ': pengaturan diagnostik kurang dari 30 resample; statistik utama tidak dihitung')
            continue
        if scope == 'BENCHMARK_FULL' and cfg['benchmark_stochastic_runs'] != 10:
            notes.append(scope + ': pengaturan diagnostik bukan 10 run stokastik; statistik utama tidak dihitung')
            continue
        complete = set(frame.dataset) == set(gd.DATASET_ORDER) and len(frame) == len(gd.DATASET_ORDER)*len(METHODS)
        expected_runs = frame.method.map(lambda method: 30 if scope == 'BENCHMARK_RESAMPLING'
                                         else (10 if method in STOCHASTIC else 1))
        if not complete or (frame.n_failures > 0).any() or not frame.n_runs.eq(expected_runs).all():
            notes.append(scope + ': statistik sepuluh dataset belum lengkap atau ada kegagalan; tidak dihitung paksa')
            continue
        for metric in METRICS:
            matrix = frame.pivot(index='dataset', columns='method', values=metric+'_mean').reindex(index=gd.DATASET_ORDER, columns=METHODS)
            if not np.isfinite(matrix.to_numpy(float)).all():
                notes.append(scope + '/' + metric + ': metrik tidak lengkap')
                continue
            ranking = matrix.rank(axis=1, ascending=metric != 'si', method='average')
            for dataset in matrix.index:
                for method in METHODS:
                    ranks.append({'source': scope, 'metric': metric, 'dataset': dataset, 'method': method,
                                  'value': matrix.loc[dataset, method], 'rank': ranking.loc[dataset, method]})
            if (matrix.nunique(axis=1) == 1).all():
                stat, p = np.nan, np.nan
                notes.append(scope + '/' + metric + ': semua metode seri; Friedman tidak terdefinisi')
            else:
                stat, p = friedmanchisquare(*(matrix[method].to_numpy(float) for method in METHODS))
            friedman.append({'source': scope, 'metric': metric, 'friedman_chi2': stat, 'p_value': p,
                             'kendall_w': stat/(len(matrix)*(len(METHODS)-1)), 'n_datasets': len(matrix), 'n_methods': len(METHODS)})
            if scope != 'BENCHMARK_RESAMPLING' or metric not in ('dbi', 'si'):
                continue
            family = []
            for method in [m for m in METHODS if m != 'CARE-KM']:
                difference = matrix['CARE-KM'] - matrix[method] if metric == 'si' else matrix[method] - matrix['CARE-KM']
                result = exact_signed_rank(difference.to_numpy(float))
                for dataset, value in difference.items():
                    audit.append({'metric': metric, 'baseline': method, 'dataset': dataset,
                                  'care_value': matrix.loc[dataset, 'CARE-KM'], 'baseline_value': matrix.loc[dataset, method],
                                  'improvement_favoring_care': value, 'zero': abs(value) <= 1e-12})
                family.append({'metric': metric, 'baseline': method, 'wins': int((difference > 1e-12).sum()),
                               'ties': int((difference.abs() <= 1e-12).sum()), 'losses': int((difference < -1e-12).sum()),
                               **{k: v for k, v in result.items() if k != 'absolute_ranks'},
                               'p_method': 'exact enumeration of nonzero signed-rank signs', 'zero_method': 'discard abs(diff)<=1e-12',
                               'absolute_rank_ties': 'average', 'alternative': 'two-sided'})
            adjusted = holm([row['p_value'] for row in family])
            for row, value in zip(family, adjusted):
                row['holm_p'] = value; pairwise.append(row)
    folder = output / 'STATISTIK_BENCHMARK'
    save_csv(folder / '01_dataset_values_and_ranks.csv', pd.DataFrame(ranks, columns=['source','metric','dataset','method','value','rank']))
    rank_frame = pd.DataFrame(ranks)
    average = (rank_frame.groupby(['source', 'metric', 'method'], as_index=False)['rank'].mean().rename(columns={'rank': 'average_rank'})
               if len(rank_frame) else pd.DataFrame(columns=['source', 'metric', 'method', 'average_rank']))
    save_csv(folder / '02_average_ranks.csv', average)
    save_csv(folder / '03_friedman_kendall.csv', pd.DataFrame(friedman, columns=['source','metric','friedman_chi2','p_value','kendall_w','n_datasets','n_methods']))
    save_csv(folder / '04_paired_wilcoxon_holm.csv', pd.DataFrame(pairwise) if pairwise else pd.DataFrame(columns=['metric','baseline','wins','ties','losses','statistic','p_value','rank_biserial','holm_p']))
    audit_frame = pd.DataFrame(audit)
    if len(audit_frame):
        audit_frame['absolute_rank'] = audit_frame.groupby(['metric', 'baseline'])['improvement_favoring_care'].transform(
            lambda x: x.abs().where(x.abs() > 1e-12).rank(method='average'))
        audit_frame['signed_rank'] = np.sign(audit_frame.improvement_favoring_care) * audit_frame.absolute_rank
    save_csv(folder / '05_paired_differences_and_ranks.csv', audit_frame if len(audit_frame) else pd.DataFrame(columns=['metric','baseline','dataset','care_value','baseline_value','improvement_favoring_care','zero','absolute_rank','signed_rank']))
    save_json(folder / 'status_statistik.json', {'notes': notes, 'independent_units': 'ten datasets',
              'primary_source': 'means of thirty matched resamples within each dataset',
              'multiple_testing': 'Holm, seven CARE-KM contrasts separately for DBI and SI',
              'runtime_ranks': 'not calculated from mixed cached/new timing measurements'})
    return notes


def run_comparison(root, bk, verify_cases, configuration, code_sources):
    root = Path(root).expanduser().resolve()
    files = Files(root)
    core, gd, constants, features = load_project(files)
    cfg = validate_config(configuration, gd)
    watched = preflight(files, cfg, gd)
    output = root / 'OUTPUT/TAMBAHAN' / STAGE
    control = root / 'OUTPUT/TAMBAHAN/KONTROL' / STAGE
    output.mkdir(parents=True, exist_ok=True); control.mkdir(parents=True, exist_ok=True)
    # Retire the previous pilot-only notebook safely, keeping its contents.
    old_notebook = root / 'PYTHON' / (OLD_STAGE + '.ipynb')
    if old_notebook.is_file():
        archive = control / 'ARSIP_NOTEBOOK' / (OLD_STAGE + '_' + uuid.uuid4().hex[:8] + '.ipynb')
        archive.parent.mkdir(parents=True, exist_ok=True)
        old_notebook.rename(archive)
        print('Notebook pilot lama dipindahkan ke arsip tahap tambahan.')
    current_runtime = runtime()
    hashes = {name: files.sha(name) for name in ['LIB/care_km.py', 'LIB/experiment_lib.py', 'LIB/generalization_data.py', 'LIB/pipeline_data.py']}
    code_state = {'project_libraries': hashes, 'embedded_sources': {k: binary_hash(v.encode('utf-8')) for k, v in code_sources.items()},
                  'notebook_code': notebook_code(files)}
    verify_path = control / 'verifikasi_kecil.json'
    verify_key = digest({'code': code_state, 'runtime': current_runtime})
    check_file = output / 'VERIFIKASI/01_kasus_kecil.csv'
    saved_verification = read_checkpoint(verify_path)
    if not cfg['force'] and saved_verification.get('key') == verify_key and check_file.is_file() and files.sha(str(check_file.relative_to(root)).replace('\\','/')) == saved_verification.get('output_sha256'):
        checks = pd.read_csv(check_file)
        print('Verifikasi kecil: gunakan hasil yang isinya terverifikasi sama.')
    else:
        checks = verify_cases(bk)
        save_csv(check_file, checks)
        save_json(verify_path, {'key': verify_key, 'output_sha256': binary_hash(check_file.read_bytes())})
    if not checks['status'].eq('LULUS').all():
        raise RuntimeError('Verifikasi kecil belum lulus.')
    print('Verifikasi kecil:', len(checks), 'pemeriksaan LULUS.')
    save_csv(output / 'PROTOKOL/01_daftar_initializer.csv', pd.DataFrame([{'method': m, 'stochastic': m in STOCHASTIC} for m in METHODS]))
    save_json(output / 'PROTOKOL/02_konfigurasi.json', {'config': cfg, 'constants': constants,
              'runtime': current_runtime, 'code': code_state, 'input_hashes': watched,
              'bk_policy': bk.IMPLEMENTATION_POLICY,
              'semantic_scope': 'saved partitions/assignments; semantic rules and ILS correspondence are evaluated in the next experiment',
              'runtime_scope': 'stored timings have origin tags; no comparative-runtime claim from mixed sessions',
              'stochastic_references': 'run 0 is the declared reference; no best-run selection'})
    legacy = Legacy(files, current_runtime, hashes)
    rows, artifacts, counts = [], [], {'CACHE': 0, 'REUSE': 0, 'FIT': 0}
    expected_jobs = planned_jobs(cfg)
    print('Jumlah run terjadwal:', sum(expected_jobs.values()), '|', expected_jobs, flush=True)
    context_number = 0
    started = time.time()
    try:
        for context in contexts(files, cfg, gd, constants, features):
            same_inputs(files, watched, code_state['notebook_code'])
            context_number += 1
            print(f"[{context_number}] {context['scope']} / {context['dataset']} / rep={context['rep']} | n={len(context['X'])}, k={context['k']}", flush=True)
            for method in METHODS:
                runs = context['stochastic_runs'] if method in STOCHASTIC else 1
                for run in range(runs):
                    row, disposition, record = execute_job(root, output, control, context, method, run,
                                                          constants, core, bk, code_state, current_runtime, files, legacy, cfg)
                    rows.append(row); counts[disposition] += 1
                    artifacts.append({'scope': context['scope'], 'dataset': context['dataset'], 'rep': context['rep'],
                                      'run': run, 'method': method, 'partition_available': row['partition_available'],
                                      'files': json.dumps(list(record.get('outputs', {})), ensure_ascii=False)})
                    if disposition == 'FIT':
                        print('  ', method, 'run', run, ': dihitung dan disimpan', flush=True)
            # Export partial tables after every complete context; JOBS remain
            # the authoritative restart point even if the session is interrupted.
            save_csv(output / '01_hasil_per_run_8_metode.csv', pd.DataFrame(rows))
            save_json(control / 'status_eksekusi.json', {'status': 'berjalan', 'completed_jobs': len(rows),
                      'counts': counts, 'last_scope': context['scope'], 'last_dataset': context['dataset']})
        same_inputs(files, watched, code_state['notebook_code'])
        raw = pd.DataFrame(rows)
        if len(raw) != sum(expected_jobs.values()):
            raise RuntimeError('Jumlah run selesai tidak sesuai konfigurasi.')
        summary = summarize(raw)
        save_csv(output / '02_ringkasan_8_metode.csv', summary)
        save_csv(output / '03_peta_mode_dan_assignment.csv', pd.DataFrame(artifacts))
        save_csv(output / '04_failure_audit.csv', raw[['scope','dataset','method','run','rep','valid','failure_flags','origin']])
        for scope in cfg['scopes']:
            save_csv(output / scope / 'hasil_per_run.csv', raw.loc[raw.scope.eq(scope)])
            save_csv(output / scope / 'ringkasan.csv', summary.loc[summary.scope.eq(scope)])
        notes = benchmark_statistics(summary, cfg, gd, output)
        full_protocol = (set(cfg['scopes']) == set(SCOPES) and cfg['benchmark_datasets'] == gd.DATASET_ORDER
                         and cfg['resample_runs'] == 30 and cfg['lms_stochastic_runs'] == 15
                         and cfg['holdout_stochastic_runs'] == 15 and cfg['benchmark_stochastic_runs'] == 10
                         and cfg['save_semantic_partitions'])
        status = {'status': 'selesai' if raw['valid'].eq(True).all() else 'selesai_dengan_kegagalan_algoritmik',
                  'completed_jobs': len(raw), 'n_valid': int(raw['valid'].eq(True).sum()),
                  'counts': counts, 'method_count': len(METHODS), 'full_protocol': full_protocol,
                  'expected_jobs_by_scope': expected_jobs,
                  'elapsed_sec': time.time()-started, 'statistics_notes': notes,
                  'reuse_reasons': legacy.reasons, 'output': str(output),
                  'semantic_ILS_evaluation': 'not part of this initializer/partition-bank experiment'}
        save_json(control / 'status_eksekusi.json', status)
        print('Tahap 25 selesai:', len(raw), 'run,', len(METHODS), 'metode. Cache/reuse/fit:', counts)
        print('Hasil:', output)
        return {'summary': summary, 'status': status, 'output': str(output)}
    except BaseException as error:
        save_json(control / 'status_eksekusi.json', {'status': 'terhenti', 'completed_jobs': len(rows), 'counts': counts,
                  'error': f'{type(error).__name__}: {error}', 'resume': 'rerun the same notebook/configuration; per-job contents are checked'})
        raise
'''

_compare25 = types.ModuleType('_compare25')
exec(compile(COMPARISON_SOURCE, '_compare25', 'exec'), _compare25.__dict__)


In [6]:
HASIL_25 = _compare25.run_comparison(
    _ROOT_25, _bk25, _verify25.small_case_verification, KONFIGURASI_25,
    {'initializer': BK_MODES_SOURCE, 'verification': VERIFICATION_SOURCE,
     'comparison': COMPARISON_SOURCE},
)
display(HASIL_25['summary'])
print(HASIL_25['status'])
print('Folder hasil:', HASIL_25['output'])


Verifikasi kecil: 55 pemeriksaan LULUS.
Jumlah run terjadwal: 3272 | {'LMS_FULL': 64, 'LMS_RESAMPLING': 240, 'LMS_TRAIN': 64, 'PENDIKOM_TRAIN': 64, 'BENCHMARK_FULL': 440, 'BENCHMARK_RESAMPLING': 2400}
[1] LMS_FULL / LMS / rep=None | n=6658, k=16
   Random run 0 : dihitung dan disimpan
   Random run 1 : dihitung dan disimpan
   Random run 2 : dihitung dan disimpan
   Random run 3 : dihitung dan disimpan
   Random run 4 : dihitung dan disimpan
   Random run 5 : dihitung dan disimpan
   Random run 6 : dihitung dan disimpan
   Random run 7 : dihitung dan disimpan
   Random run 8 : dihitung dan disimpan
   Random run 9 : dihitung dan disimpan
   Random run 10 : dihitung dan disimpan
   Random run 11 : dihitung dan disimpan
   Random run 12 : dihitung dan disimpan
   Random run 13 : dihitung dan disimpan
   Random run 14 : dihitung dan disimpan
   Huang run 0 : dihitung dan disimpan
   Huang run 1 : dihitung dan disimpan
   Huang run 2 : dihitung dan disimpan
   Huang run 3 : dihitung dan di

,scope,dataset,method,n_runs,n_valid,n_failures,partition_runs,initial_cost_mean,initial_cost_sd,final_cost_mean,final_cost_sd,dbi_mean,dbi_sd,si_mean,si_sd,iterations_mean,iterations_sd
0,LMS_FULL,LMS,Random,15,15,0,15,26071.733333,584.288758,23887.466667,342.219826,4.277627,1.513816,0.080822,0.019609,3.466667,0.743223
1,LMS_FULL,LMS,Huang,15,15,0,15,27892.533333,700.906440,24244.666667,429.368418,3.415911,1.244482,0.085299,0.014256,4.000000,1.195229
2,LMS_FULL,LMS,Cao,1,1,0,1,26981.000000,NaN,24701.000000,NaN,1.562390,NaN,0.110500,NaN,3.000000,NaN
3,LMS_FULL,LMS,Bai,1,1,0,1,26291.000000,NaN,24123.000000,NaN,1.680321,NaN,0.102853,NaN,3.000000,NaN
4,LMS_FULL,LMS,K-Modes++,15,15,0,15,25953.133333,910.009644,24356.466667,492.876523,2.598695,0.563971,0.098339,0.016368,3.466667,0.915475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,BENCHMARK_RESAMPLING,nursery,Bai,30,30,0,0,39191.700000,58.912062,39191.700000,58.912062,1.292556,0.002721,0.056195,0.002357,1.000000,0.000000
188,BENCHMARK_RESAMPLING,nursery,K-Modes++,30,30,0,0,40395.566667,597.761438,40374.566667,570.350992,2.163332,0.528622,0.056950,0.003532,1.033333,0.182574
189,BENCHMARK_RESAMPLING,nursery,HR-2023,30,30,0,0,40879.600000,652.085810,40662.433333,597.196651,2.354697,0.698246,0.054672,0.004422,1.200000,0.406838
190,BENCHMARK_RESAMPLING,nursery,CARE-KM,30,30,0,0,38893.433333,52.925245,38893.433333,52.925245,1.270617,0.014842,0.059670,0.001982,1.000000,0.000000


{'status': 'selesai', 'completed_jobs': 3272, 'n_valid': 3272, 'counts': {'CACHE': 0, 'REUSE': 2530, 'FIT': 742}, 'method_count': 8, 'full_protocol': True, 'expected_jobs_by_scope': {'LMS_FULL': 64, 'LMS_RESAMPLING': 240, 'LMS_TRAIN': 64, 'PENDIKOM_TRAIN': 64, 'BENCHMARK_FULL': 440, 'BENCHMARK_RESAMPLING': 2400}, 'elapsed_sec': 1020.8037414550781, 'statistics_notes': [], 'reuse_reasons': {}, 'output': '/content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/25_Perbandingan_Initializer_Terintegrasi', 'semantic_ILS_evaluation': 'not part of this initializer/partition-bank experiment'}
Folder hasil: /content/drive/MyDrive/REVISED/OUTPUT/TAMBAHAN/25_Perbandingan_Initializer_Terintegrasi


## Membaca hasil

- `01_hasil_per_run_8_metode.csv`: metrik, seed, kelompok data, dan asal
  setiap hasil (`COMPUTED` atau `REUSED_VERIFIED`).
- `02_ringkasan_8_metode.csv`: mean, simpangan baku, jumlah run valid,
  kegagalan, dan ketersediaan partisi per metode/kelompok/dataset.
- `03_peta_mode_dan_assignment.csv`: lokasi bank mode dan assignment
  untuk analisis semantik berikutnya.
- `04_failure_audit.csv`: denominator dan status kegagalan setiap run.
- `BANK/`: mode awal/akhir, assignment train/test, dan trace BK-Modes.
- `STATISTIK_BENCHMARK/`: nilai dan rank sepuluh dataset, average rank,
  Friedman/Kendall, paired differences, dan Wilcoxon/Holm untuk delapan metode.
- `PROTOKOL/`: versi kode, konfigurasi, runtime, dan hash masukan.

Statistik utama benchmark menggunakan mean 30 resample pada masing-masing
dari sepuluh dataset. Wilcoxon memakai enumerasi tepat tanda nonzero dengan
tied absolute ranks rata-rata dan toleransi nol 1e-12; Holm diterapkan pada
tujuh kontras CARE-KM secara terpisah untuk DBI dan SI. Perbedaan p-value
atau rank setelah menambah metode merupakan hasil analisis baru.

Pengaturan diagnostik yang mengurangi kelompok/dataset/pengulangan diberi
status `full_protocol=False`; statistik utama yang belum memenuhi sepuluh
dataset dan 30 resample tidak dipaksakan. Sesi terputus dapat dilanjutkan
melalui checkpoint per run dengan konfigurasi yang sama.
